# 01 Metadata EDA

이 노트북은 `raw_metadata.parquet`, `category_enriched_metadata.parquet`를 확인하기 위한 EDA 노트북입니다.

목적:
- 전체 데이터 개수 확인
- 음식명/음식코드 분포 확인
- 업종/상품군 매핑 결과 확인
- fallback_default 항목 확인
- 최종 DB 생성 전 데이터 상태 점검


In [ ]:

from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path("..").resolve()
RAW_METADATA_PATH = ROOT / "data" / "metadata" / "raw_metadata.parquet"
CATEGORY_METADATA_PATH = ROOT / "data" / "metadata" / "category_enriched_metadata.parquet"

RAW_METADATA_PATH, CATEGORY_METADATA_PATH


In [ ]:

raw_df = pd.read_parquet(RAW_METADATA_PATH)
print("raw_df shape:", raw_df.shape)
raw_df.head()


In [ ]:

raw_df.info()


In [ ]:

raw_df[[
    "source_file_name",
    "original_food_name",
    "product_name",
    "food_code",
    "image_path",
    "image_width",
    "image_height",
    "image_size_bytes",
]].head(20)


In [ ]:

summary = {
    "row_count": len(raw_df),
    "unique_food_name_count": raw_df["original_food_name"].nunique(),
    "unique_food_code_count": raw_df["food_code"].nunique(),
    "missing_image_count": raw_df["image_path"].isna().sum(),
    "food_name_missing_count": (raw_df["original_food_name"].fillna("").astype(str).str.strip() == "").sum(),
    "total_image_size_gb": raw_df["image_size_bytes"].sum() / (1024 ** 3),
    "avg_image_size_mb": raw_df["image_size_bytes"].mean() / (1024 ** 2),
}
summary


In [ ]:

food_name_dist = (
    raw_df["original_food_name"]
    .value_counts()
    .reset_index()
)
food_name_dist.columns = ["original_food_name", "count"]
food_name_dist.head(30)


In [ ]:

top_n = 30

plot_df = food_name_dist.head(top_n).sort_values("count")

plt.figure(figsize=(10, 8))
plt.barh(plot_df["original_food_name"], plot_df["count"])
plt.title(f"Top {top_n} Food Names")
plt.xlabel("Image Count")
plt.ylabel("Food Name")
plt.tight_layout()
plt.show()


In [ ]:

food_code_dist = (
    raw_df["food_code"]
    .value_counts()
    .reset_index()
)
food_code_dist.columns = ["food_code", "count"]
food_code_dist.head(30)


In [ ]:

plt.figure(figsize=(10, 8))
plot_df = food_code_dist.head(30).sort_values("count")
plt.barh(plot_df["food_code"], plot_df["count"])
plt.title("Top 30 Food Codes")
plt.xlabel("Image Count")
plt.ylabel("Food Code")
plt.tight_layout()
plt.show()


In [ ]:

if CATEGORY_METADATA_PATH.exists():
    category_df = pd.read_parquet(CATEGORY_METADATA_PATH)
    print("category_df shape:", category_df.shape)
else:
    category_df = None
    print("category_enriched_metadata.parquet does not exist.")


In [ ]:

if category_df is not None:
    display(category_df[[
        "original_food_name",
        "food_code",
        "business_category",
        "product_group",
        "category_mapping_status",
        "category_matched_keyword",
    ]].head(30))


In [ ]:

if category_df is not None:
    status_dist = category_df["category_mapping_status"].value_counts().reset_index()
    status_dist.columns = ["category_mapping_status", "count"]
    status_dist["ratio"] = status_dist["count"] / len(category_df)
    display(status_dist)


In [ ]:

if category_df is not None:
    business_dist = category_df["business_category"].value_counts().reset_index()
    business_dist.columns = ["business_category", "count"]
    business_dist["ratio"] = business_dist["count"] / len(category_df)
    display(business_dist)

    plt.figure(figsize=(8, 5))
    plt.bar(business_dist["business_category"], business_dist["count"])
    plt.title("Business Category Distribution")
    plt.xlabel("Business Category")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:

if category_df is not None:
    product_group_dist = category_df["product_group"].value_counts().reset_index()
    product_group_dist.columns = ["product_group", "count"]
    product_group_dist["ratio"] = product_group_dist["count"] / len(category_df)
    display(product_group_dist.head(50))

    plt.figure(figsize=(10, 8))
    plot_df = product_group_dist.head(30).sort_values("count")
    plt.barh(plot_df["product_group"], plot_df["count"])
    plt.title("Top Product Groups")
    plt.xlabel("Count")
    plt.ylabel("Product Group")
    plt.tight_layout()
    plt.show()


In [ ]:

if category_df is not None:
    fallback_df = category_df[
        category_df["category_mapping_status"] == "fallback_default"
    ][[
        "original_food_name",
        "product_name",
        "food_code",
        "business_category",
        "product_group",
        "category_mapping_status",
    ]].drop_duplicates().sort_values("original_food_name")

    print("fallback unique food count:", len(fallback_df))
    display(fallback_df.head(100))


In [ ]:

if category_df is not None:
    sample_df = (
        category_df
        .sample(min(20, len(category_df)), random_state=42)
        [[
            "original_food_name",
            "food_code",
            "business_category",
            "product_group",
            "image_path",
        ]]
    )
    display(sample_df)
